In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Residual Networks

On ImageNet, depth and accuracy had so far grown together, from the eight
layers of AlexNet to the 22 of GoogLeNet, and the batch normalization of
that section made such depth easier to train. We might expect
that adding still more layers keeps helping, at worst until the network
starts to overfit. @He.Zhang.Ren.ea.2016 tested this expectation.
They trained *plain* networks, stacks of convolutions with batch
normalization and ReLU in the style of VGG, at several depths. On CIFAR-10 a
plain network with 56 layers ended training with a higher *training* error
than one with 20 layers. A deeper network that fits even its training data
worse does not overfit; the optimizer fails to find a good setting of its
parameters. This *degradation problem* is puzzling, because the deeper
network could copy the shallower one and pass its result through the extra
layers unchanged, so a setting that is at least as good exists. Why does
training not find it, and how can we change the network so that it does?

*Residual networks* (ResNets) make that setting easy to reach. Each block
adds a learned correction to its input instead of replacing it, so a block
whose correction is zero passes its input through unchanged. This small
change was enough. On ImageNet, where a plain network with 34 layers trained
worse than one with 18, the residual network with 34 layers was almost three
points more accurate than its 18-layer relative, and residual networks with
152 layers won the ImageNet challenge of 2015.

We first ask what the extra layers of a deeper network would have to do in
order not to hurt, and build the residual block. Then we repeat the
experiment ourselves: on Fashion-MNIST we train plain and residual networks
of 20 and 56 layers and watch the degradation appear and disappear. After
that we build and train the 18-layer ResNet. Two variants close the section:
ResNeXt makes the block wider at the same cost, and DenseNet concatenates
where ResNet adds.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Function Classes and Depth

It is instructive to describe a network by the set of functions it can
represent as its parameters vary, its *function class* $\mathcal{F}$.
Training searches $\mathcal{F}$ for the function that best fits the training
data. When does a larger architecture, with class $\mathcal{F}'$, have a
better function to offer? Certainly if the classes are *nested*,
$\mathcal{F}\subseteq\mathcal{F}'$: the larger class still contains the best
function of the smaller one, so the smallest training loss available in it
can only be lower. For classes that are not nested there is no such
guarantee, since the larger class may have lost the function that worked.
the figure illustrates the difference, with a star for
the function we would like to find. Note, however, that nesting is a
statement about the best function in each class. It does not promise that
the optimizer finds that function, nor that the function generalizes better.

![Function classes of increasing size. On the left the classes are not nested: a larger class need not contain the functions of a smaller one, and need not come closer to the target $f^*$. On the right each class contains the previous ones, so the best function available can only improve as the class grows.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/functionclasses.svg)

Adding a layer to a network nests the function classes if the new layer can
compute the identity $f(\mathbf{x}) = \mathbf{x}$: the deeper network then
contains every function of the shallower one. A plain layer
$\mathbf{x} \mapsto \textrm{ReLU}(\textrm{BN}(\mathbf{W}\mathbf{x} +
\mathbf{b}))$ can compute the identity on the nonnegative inputs that it
receives, but only for particular values of its weights. So the function
classes of plain networks are nested as well, and plain and residual
networks both belong in the right panel of the figure. Nesting, then, is not
what the plain networks lack. What the degradation results show is that
gradient descent does not find those particular values, and nothing in the
plain layer draws it toward them. A residual block changes the
parameterization so that the identity lies at the simplest point of the
parameter space, where the correction is zero.

## Residual Blocks

the figure compares a regular block with a residual one.
Both receive the input $\mathbf{x}$ and feed their result into an activation
function. The regular block on the left must learn the desired mapping
$f(\mathbf{x})$ directly. The residual block on the right learns the
*residual mapping* $g(\mathbf{x}) = f(\mathbf{x}) - \mathbf{x}$ and adds the
input back, so its output is $\mathbf{x} + g(\mathbf{x})$. If the identity
is the desired mapping, the residual block only has to drive $g$ to zero,
which it can do in its last layer alone, whatever the layers before it
compute. The line that carries $\mathbf{x}$ to the addition is called a
*residual connection* or *shortcut connection*.

![A regular block (left) must learn the mapping $f(\mathbf{x})$ directly. A residual block (right) learns the residual mapping $g(\mathbf{x}) = f(\mathbf{x}) - \mathbf{x}$ and adds the input back, so the identity mapping $f(\mathbf{x}) = \mathbf{x}$ corresponds to $g(\mathbf{x}) = 0$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-residual-concept.svg)

The shortcut also changes how the gradient travels. Let us stack two such
blocks, $\mathbf{x}_1 = \mathbf{x}_0 + g_1(\mathbf{x}_0)$ and
$\mathbf{x}_2 = \mathbf{x}_1 + g_2(\mathbf{x}_1)$. Then
$\mathbf{x}_2 = \mathbf{x}_0 + g_1(\mathbf{x}_0) + g_2(\mathbf{x}_1)$: the
output contains the input itself, plus two corrections. Its Jacobian with
respect to $\mathbf{x}_0$ is
$(\mathbf{I} + \partial g_2 / \partial \mathbf{x}_1)
(\mathbf{I} + \partial g_1 / \partial \mathbf{x}_0)$, which multiplies out
to the identity plus terms that pass through one branch or through both. The
same holds for any number of blocks, so the gradient of the loss reaches
every block through the chain of identity terms, without passing through the
convolutions of the blocks above it. A plain network has no such term: the
gradient reaches a layer only through all the layers above it.

ResNet itself is not quite this clean. Two things remain on the identity
path: the ReLU after each addition, which blocks the gradient wherever the
sum is negative, and the $1 \times 1$ convolutions, with their batch
normalization, on the shortcuts of the blocks that change the shape. The
pre-activation design of the exercises removes the ReLU.

How much do these two arguments explain? They make it plausible that
residual networks are easy to optimize, and
that section checks that they are. They do not show why
plain networks fail. Vanishing gradients, the obvious suspect, are an unlikely cause:
@He.Zhang.Ren.ea.2016 report that the gradients of their plain
networks, which use batch normalization, kept a healthy size, and the
networks degraded all the same. As with batch normalization, the remedy is
better understood than the disease.

ResNet's residual block follows VGG's design of $3 \times 3$ convolutions. It
has two $3 \times 3$ convolutional layers with the same number of output
channels. Each convolution is followed by batch normalization, and the first
also by a ReLU. The convolutions have no bias, since the batch normalization
after them would remove it (that section). The block adds its
input to the output of the second batch normalization and applies a final
ReLU.

When is this block the identity? The last layer of its branch is a batch
normalization, so the branch outputs zero when the scale $\gamma$ and the
shift $\beta$ of that layer are zero. Because the final ReLU sets negative
values to zero, the block is then the identity only on inputs that are
already nonnegative. Every block of the network receives such inputs, since
each one follows a ReLU. Note that the block does not start out this way:
$\gamma$ is initialized to 1, so the argument concerns what is easy to
reach, not where training begins. Some later training recipes do start this
$\gamma$ at zero [@He.Zhang.Zhang.ea.2019]; we keep it at 1.

The output of the branch must have the shape of the input so that the two
can be added. When a block changes the number of channels or halves the
resolution, a $1 \times 1$ convolution with the same stride, followed by
batch normalization, transforms the input on the shortcut to the new shape.
This is rather crude: with stride 2, the convolution reads only one of every
four input positions. An exercise repairs it. The class `Residual`
implements both the identity and the $1 \times 1$ shortcut.

In [ ]:
class Residual(nnx.Module):
    """The Residual block of ResNet models."""
    def __init__(self, num_channels, use_1x1conv=False, strides=(1, 1),
                 in_channels=None, rngs=None):
        in_channels = num_channels if in_channels is None else in_channels
        strides = (strides, strides) if isinstance(strides, int) else strides
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        # He initialization; no conv biases, since the batch normalization
        # after each conv removes them
        he = nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal')
        # padding=1 on every side: 'SAME' pads only right and bottom when the
        # stride is 2 and the sizes are even
        self.conv1 = nnx.Conv(in_channels, num_channels, kernel_size=(3, 3),
                              padding=1, strides=strides, use_bias=False,
                              kernel_init=he, rngs=rngs)
        self.conv2 = nnx.Conv(num_channels, num_channels, kernel_size=(3, 3),
                              padding='same', use_bias=False,
                              kernel_init=he, rngs=rngs)
        # Running averages decay 0.9 and a two-pass variance (the Flax
        # defaults are 0.99 and a one-pass variance)
        bn = lambda: nnx.BatchNorm(num_channels, momentum=0.9,
                                   use_fast_variance=False, rngs=rngs)
        # Auto-enable 1x1 conv when downsampling so the residual shape matches.
        if use_1x1conv or any(s != 1 for s in strides):
            self.conv3 = nnx.Conv(in_channels, num_channels,
                                  kernel_size=(1, 1), strides=strides,
                                  use_bias=False, kernel_init=he, rngs=rngs)
            self.bn3 = bn()
        else:
            self.conv3 = None
        self.bn1, self.bn2 = bn(), bn()

    def __call__(self, X):
        Y = nnx.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.bn3(self.conv3(X))
        Y += X
        return nnx.relu(Y)

NNX layers need the number of input channels when they are constructed, so
the JAX `Residual` takes `in_channels`, which defaults to `num_channels`.

the figure shows the two variants: `use_1x1conv=True`, or a
stride other than 1, selects the one on the right.

![The two variants of the ResNet block: with an identity shortcut when input and output shapes match (left), and with a $1 \times 1$ convolution and batch normalization on the shortcut that adjust channels and resolution so that the addition is defined (right).](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-resnet-block.svg)

A block with matching shapes maps a minibatch of four three-channel
$6 \times 6$ inputs to an output of the same shape.

In [ ]:
blk = Residual(3)
X = jax.random.normal(d2l.get_key(), (4, 6, 6, 3))
blk(X).shape

With six output channels and stride 2, the block doubles the channels and
halves the height and the width. This is the block that opens each later
stage of ResNet below.

In [ ]:
blk = Residual(6, use_1x1conv=True, strides=(2, 2), in_channels=3)
blk(X).shape

## Plain and Residual Networks Compared

Does the degradation problem appear on Fashion-MNIST, and does the residual
block remove it? To find out we need networks that differ in the shortcut
alone. We take the design that @He.Zhang.Ren.ea.2016 used on
CIFAR-10 and apply it to the $28 \times 28$ images of Fashion-MNIST, at
which depth is cheap. A *plain block* has the layers of `Residual` without
the shortcut: two $3 \times 3$ convolutions, each followed by batch
normalization and a ReLU.

In [ ]:
def plain_block(num_channels, strides, in_channels, rngs):
    """The layers of `Residual` without the shortcut."""
    he = nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal')
    bn = lambda: nnx.BatchNorm(num_channels, momentum=0.9,
                               use_fast_variance=False, rngs=rngs)
    return nnx.Sequential(
        nnx.Conv(in_channels, num_channels, kernel_size=(3, 3), padding=1,
                 strides=strides, use_bias=False, kernel_init=he, rngs=rngs),
        bn(), nnx.relu,
        nnx.Conv(num_channels, num_channels, kernel_size=(3, 3), padding=1,
                 use_bias=False, kernel_init=he, rngs=rngs),
        bn(), nnx.relu)

The network starts with a $3 \times 3$ convolution with 16 output channels,
batch normalization, and a ReLU. Three stages of $n$ blocks follow, with 16,
32, and 64 channels; the first block of the second and of the third stage
halves the height and the width. Global average pooling and a fully
connected layer form the head. With two convolutional layers per block, the
network has $6n + 2$ layers: 20 for $n = 3$ and 56 for $n = 9$. The
argument `block` is `plain_block` or `Residual`. As in the earlier
networks of this chapter, the weights start from He initialization
(that section), and the learning rate follows the warmup and
cosine schedule (`d2l.cosine_warmup` of that section). The peak is
0.1, the rate at which several networks of that section were at the
edge of stability. With a normalization after every convolution it is safe,
as that section led us to expect.

In [ ]:
class DeepNet(d2l.Classifier):
    """6n + 2 layers: a 3x3 stem, three stages of n blocks, a linear head."""
    def __init__(self, block, n, lr=0.1, num_steps=2345, num_classes=10,
                 rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['block', 'rngs'])
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        he = nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal')
        # Fashion-MNIST images have one channel
        layers = [nnx.Conv(1, 16, kernel_size=(3, 3), padding=1,
                           use_bias=False, kernel_init=he, rngs=rngs),
                  nnx.BatchNorm(16, momentum=0.9, use_fast_variance=False,
                                rngs=rngs), nnx.relu]
        in_channels = 16
        for i, num_channels in enumerate((16, 32, 64)):
            for j in range(n):  # The first block of stages 2 and 3 downsamples
                strides = 2 if i > 0 and j == 0 else 1
                layers.append(block(num_channels, strides=strides,
                                         in_channels=in_channels, rngs=rngs))
                in_channels = num_channels
        layers += [global_avg_pool,
                   nnx.Linear(64, num_classes, kernel_init=he, rngs=rngs)]
        self.net = nnx.Sequential(*layers)

    def configure_optimizers(self):
        return optax.sgd(lambda t: self.lr * d2l.cosine_warmup(
            t, self.num_steps, self.num_steps // 10))

def global_avg_pool(X):  # Average over height and width (NHWC)
    return X.mean(axis=(1, 2))

NNX layers draw their weights when they are constructed, so every layer,
including those inside `Residual`, receives He initialization
(`variance_scaling(2.0, 'fan_in', 'normal')`) as its `kernel_init`.

We train the plain and the residual network at both depths with identical
settings, minibatches of 128 images and five epochs, and keep the history
of each run.

In [ ]:
data = d2l.FashionMNIST(batch_size=128)
num_steps = 5 * len(data.train_dataloader())
history = {}
for name, block in (('plain', plain_block), ('residual', Residual)):
    for n in (3, 9):
        model = DeepNet(block, n, lr=0.1, num_steps=num_steps)
        trainer = d2l.Trainer(max_epochs=5, num_gpus=1)
        trainer.board.display = False  # We plot the losses ourselves
        trainer.fit(model, data)
        history[f'{name}, {6 * n + 2} layers'] = trainer.history

The training losses of the four networks share one plot, on a logarithmic
scale; the 20-layer networks are blue, the 56-layer ones red, and the plain
networks dashed.

In [ ]:
d2l.plot(list(range(1, 6)), [h['train_loss'] for h in history.values()],
         'epoch', 'training loss', legend=list(history), yscale='log',
         fmts=('b--', 'r--', 'b-', 'r-'))

In [ ]:
{name: (round(h['train_loss'][-1], 3), round(h['val_acc'][-1], 3))
 for name, h in history.items()}  # Training loss, validation accuracy

The plain networks show the degradation problem. The 56-layer one ends
with a training loss of about 0.25 to 0.3, above the 0.2 or so of the
20-layer one, and its curve lies above the shallower network's in every
epoch. It fits even its training data worse, so overfitting cannot be the
cause. With residual blocks the order reverses: the 56-layer network ends at
about 0.15 to 0.17, below the 20-layer one, which ends near 0.19 with or
without shortcuts. At 20 layers the shortcut makes no visible difference.
At 56 it decides whether 36 extra layers are a burden or a gain, and since
the four networks differ in nothing but depth and the shortcut, there is no
other explanation to reach for. Validation accuracy shows the same
degradation more faintly: the deep plain network ends lowest, at 88% to 90%,
while the other three end between 90.5% and 91.5%, too close together to
order after five epochs.

## ResNet-18

### Stem, Stages, and Head

ResNet arranges residual blocks into the stem, body, and head of GoogLeNet
(that section). The stem is GoogLeNet's: a $7 \times 7$
convolution with 64 output channels and stride 2, followed by $3 \times 3$
max-pooling with stride 2, which together reduce the resolution fourfold.
ResNet adds batch normalization after the convolution. The method `stem`
builds it.

In [ ]:
class ResNet(d2l.Classifier):
    def __init__(self, arch, lr=0.1, num_steps=4690, num_classes=10,
                 in_channels=1, rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs'])
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.net = self.create_net(in_channels, rngs)

    def stem(self, in_channels, rngs):
        he = nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal')
        return nnx.Sequential(
            nnx.Conv(in_channels, 64, kernel_size=(7, 7), strides=(2, 2),
                     padding=3, use_bias=False, kernel_init=he, rngs=rngs),
            nnx.BatchNorm(64, momentum=0.9, use_fast_variance=False,
                          rngs=rngs), nnx.relu, max_pool)

def max_pool(X):  # 3x3 windows, stride 2, padding 1 on every side
    return nnx.max_pool(X, window_shape=(3, 3), strides=(2, 2),
                        padding=((1, 1), (1, 1)))

The body consists of four *stages*, each a sequence of residual blocks with
the same number of output channels. The first stage keeps the 64 channels
and the resolution of the stem, which has already reduced the resolution.
The first block of each later stage doubles the number of channels and
halves the height and the width. The method `stage` builds one stage, of
`num_residuals` blocks.

In [ ]:
@d2l.add_to_class(ResNet)
def stage(self, num_residuals, num_channels, in_channels,
          first_block=False, rngs=None):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(num_channels, use_1x1conv=True,
                                strides=(2, 2), in_channels=in_channels,
                                rngs=rngs))
        else:
            blk.append(Residual(num_channels, in_channels=in_channels,
                                rngs=rngs))
        in_channels = num_channels
    return nnx.Sequential(*blk)

The head is that of NiN and GoogLeNet: global average pooling, followed by a
fully connected output layer. The argument `arch` lists the number of blocks
and the number of channels of each stage. As in `DeepNet`, the weights
start from He initialization. The only bias, that of the output layer,
starts at zero.

In [ ]:
@d2l.add_to_class(ResNet)
def create_net(self, in_channels, rngs):
    layers = [self.stem(in_channels, rngs)]
    stage_channels = 64
    for i, (num_residuals, num_channels) in enumerate(self.arch):
        layers.append(self.stage(num_residuals, num_channels, stage_channels,
                                 first_block=(i == 0), rngs=rngs))
        stage_channels = num_channels
    layers.append(nnx.Sequential(
        global_avg_pool,
        nnx.Linear(stage_channels, self.num_classes, rngs=rngs,
                   kernel_init=nnx.initializers.variance_scaling(
                       2.0, 'fan_in', 'normal'))))
    return nnx.Sequential(*layers)

With two blocks in each of the four stages we obtain ResNet-18. The name
counts layers: each stage holds four $3 \times 3$ convolutional layers, not
counting the $1 \times 1$ convolutions on the shortcuts. Together with the
$7 \times 7$ convolution of the stem and the fully connected output layer,
these are 18 layers. ResNet-34 has 3, 4, 6, and 3 blocks per stage. ResNet-50, -101, and -152
replace the block by a *bottleneck* block of a $1 \times 1$, a $3 \times 3$,
and a $1 \times 1$ convolution, the ResNeXt block below with a single
group. the figure shows ResNet-18.

![ResNet-18: a $7\times 7$ convolutional stem, four stages of two residual blocks each, where the first block of stages 2, 3, and 4 halves the resolution and doubles the channels, and a head of global average pooling and one fully connected layer. Shapes are those of a $96 \times 96$ input.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-resnet18.svg)

In [ ]:
class ResNet18(ResNet):
    def __init__(self, lr=0.1, num_steps=4690, num_classes=10, in_channels=1,
                 rngs=None):
        super().__init__(((2, 64), (2, 128), (2, 256), (2, 512)),
                         lr, num_steps, num_classes, in_channels, rngs)

We follow the shape of a single-channel $96 \times 96$ image through the
network. As before, the resolution shrinks and the number of channels grows
until global average pooling collapses each channel to one number.

In [ ]:
ResNet18().layer_summary((1, 96, 96, 1))

### Training

We train ResNet-18 on Fashion-MNIST, upsampled to $96 \times 96$ pixels,
with minibatches of 128 images and 10 epochs. It borrows the optimizer and
the schedule of `DeepNet`, peak 0.1 included; with batch normalization in
every block, that peak is safe.

In [ ]:
ResNet.configure_optimizers = DeepNet.configure_optimizers  # Same schedule

In [ ]:
data = d2l.FashionMNIST(batch_size=128, resize=(96, 96))
model = ResNet18(lr=0.1, num_steps=10 * len(data.train_dataloader()))
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
trainer.fit(model, data)

In [ ]:
round(trainer.history['val_acc'][-1], 3)  # Validation accuracy after 10 epochs

After 10 epochs ResNet-18 reaches a validation accuracy of about 93%, and
its last three epochs change it by less than a point. That equals the
quarter-width VGG, the best network of that section, with a quarter
of its parameters and at $96 \times 96$ instead of $224 \times 224$ pixels.
We should not read more into the comparison than it can bear. The two
networks differ in architecture, normalization, and input size at once, so
the run does not isolate what the residual connection contributes. The
paired runs of that section do.

The training loss of ResNet-18 falls below 0.01, far below its validation
loss of about 0.27, which stops improving after the first few epochs: with
11.2 million parameters for 60,000 training images, the network fits the
training data almost perfectly and new data much less well. More data, data
augmentation, and regularization reduce this gap; the training recipes of
that section combine them.

## ResNeXt and DenseNet

The residual block fixes how a block passes its input on: the input is added
to the block's output. Two variants keep this idea and change the block.
ResNeXt changes what the block computes, so that it can be wider at the same
cost; DenseNet changes how the input is passed on, by concatenation instead
of addition. We use the ResNeXt block again when we search over network
designs in that section.

### ResNeXt

More channels make a block more expressive, but the cost of a convolution
that maps $c_\textrm{i}$ input channels to $c_\textrm{o}$ output
channels grows with the product $c_\textrm{i} c_\textrm{o}$
(that section): doubling the width of a block quadruples its
arithmetic. ResNeXt [@Xie.Girshick.Dollar.ea.2017] borrows from the
Inception block of that section, which processes its input in
separate branches. Unlike Inception, it makes all branches identical.

A *grouped convolution* splits the $c_\textrm{i}$ input channels into $g$
groups of $c_\textrm{i}/g$ channels and convolves each group separately into
$c_\textrm{o}/g$ output channels. Each group has a weight tensor of size
$(c_\textrm{i}/g) \times (c_\textrm{o}/g) \times k \times k$, so the $g$
groups together have $c_\textrm{i} c_\textrm{o} k^2 / g$ weights instead of
$c_\textrm{i} c_\textrm{o} k^2$, and the layer performs $g$ times fewer
multiply-adds. We assume that $g$ divides both $c_\textrm{i}$ and
$c_\textrm{o}$. The measured speedup is usually smaller than $g$, since
multiply-adds count arithmetic and not time (that section).
Grouped convolutions first appeared in AlexNet
[@Krizhevsky.Sutskever.Hinton.2012], which split three of its five
convolutional layers into two groups, one per GPU.

The groups of a grouped convolution exchange no information. The ResNeXt
block in the figure mixes them by placing the grouped
$3 \times 3$ convolution between two $1 \times 1$ convolutions. The first
reduces the $c$ channels of the input to $b$ *bottleneck* channels, and the
second restores $c$ channels. The two $1 \times 1$ convolutions have $2cb$
weights and the grouped $3 \times 3$ convolution $9b^2/g$. The shortcut is
that of `Residual`. The stride of a downsampling block sits on the
$3 \times 3$ convolution, not on the first $1 \times 1$ one.

What does "wider at the same cost" amount to? A bottleneck block in the
first stage of ResNet-50 has $c = 256$, $b = 64$, and a single group, hence
$2 \cdot 256 \cdot 64 + 9 \cdot 64^2 = 69{,}632$ weights. The block of
the figure, with $b = 128$ and $g = 32$, has
$2 \cdot 256 \cdot 128 + 9 \cdot 128^2/32 = 70{,}144$: twice the width in
the middle for less than 1% more weights. On ImageNet this exchange lowers
the top-1 error of ResNet-50 from 23.9% to 22.2%
[@Xie.Girshick.Dollar.ea.2017].

![The ResNeXt block: a grouped $3 \times 3$ convolution with $\mathit{g}$ groups between two $1 \times 1$ convolutions that mix information across the groups. The block shown has $\mathit{c} = 256$ channels, $\mathit{b} = 128$ bottleneck channels, and $\mathit{g} = 32$ groups of four channels.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-resnext-block.svg)

The class `ResNeXtBlock` takes the number of output channels `num_channels`
($c$), the number of groups `groups` ($g$), and `bot_mul`, the ratio $b/c$
of bottleneck channels to output channels. As in `Residual`,
`strides=2` halves the height and the width and switches on the
$1 \times 1$ convolution on the shortcut.

In [ ]:
class ResNeXtBlock(nnx.Module):
    """The ResNeXt block."""
    def __init__(self, num_channels, groups, bot_mul, use_1x1conv=False,
                 strides=(1, 1), in_channels=None, rngs=None):
        in_channels = num_channels if in_channels is None else in_channels
        strides = (strides, strides) if isinstance(strides, int) else strides
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        bot_channels = int(round(num_channels * bot_mul))
        # He initialization; no conv biases, since the batch normalization
        # after each conv removes them
        he = nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal')
        self.conv1 = nnx.Conv(in_channels, bot_channels, kernel_size=(1, 1),
                              strides=(1, 1), use_bias=False,
                              kernel_init=he, rngs=rngs)
        self.conv2 = nnx.Conv(bot_channels, bot_channels,
                              kernel_size=(3, 3), strides=strides,
                              padding=1, feature_group_count=groups,
                              use_bias=False, kernel_init=he, rngs=rngs)
        self.conv3 = nnx.Conv(bot_channels, num_channels, kernel_size=(1, 1),
                              strides=(1, 1), use_bias=False,
                              kernel_init=he, rngs=rngs)
        # Running averages decay 0.9 and a two-pass variance (the Flax
        # defaults are 0.99 and a one-pass variance)
        bn = lambda n: nnx.BatchNorm(n, momentum=0.9, use_fast_variance=False,
                                     rngs=rngs)
        self.bn1, self.bn2 = bn(bot_channels), bn(bot_channels)
        self.bn3 = bn(num_channels)
        # Auto-enable 1x1 conv when downsampling so the residual shape matches.
        if use_1x1conv or any(s != 1 for s in strides):
            self.conv4 = nnx.Conv(in_channels, num_channels,
                                  kernel_size=(1, 1), strides=strides,
                                  use_bias=False, kernel_init=he, rngs=rngs)
            self.bn4 = bn(num_channels)
        else:
            self.conv4 = None

    def __call__(self, X):
        Y = nnx.relu(self.bn1(self.conv1(X)))
        Y = nnx.relu(self.bn2(self.conv2(Y)))
        Y = self.bn3(self.conv3(Y))
        if self.conv4:
            X = self.bn4(self.conv4(X))
        return nnx.relu(Y + X)

A block with 32 channels, 16 groups, and as many bottleneck channels as
output channels (`bot_mul=1`) keeps the shape of its input.

In [ ]:
blk = ResNeXtBlock(32, 16, 1)
X = jax.random.normal(d2l.get_key(), (4, 96, 96, 32))
blk(X).shape

### DenseNet

A residual block computes $\mathbf{x} + g(\mathbf{x})$: its output keeps the
number of channels of its input, and what the block adds is mixed into the
input. *DenseNet* (dense convolutional network)
[@Huang.Liu.Van-Der-Maaten.ea.2017] instead *concatenates* the output of
each layer to its input along the channel axis. Where a stack of residual
blocks computes $\mathbf{x}_k = \mathbf{x}_{k-1} + g_k(\mathbf{x}_{k-1})$, a
sequence of dense layers $f_1, f_2, \ldots$ computes

$$
\mathbf{x}_k = \left[\mathbf{x}_{k-1}, f_k(\mathbf{x}_{k-1})\right],
$$

where $[\cdot, \cdot]$ denotes concatenation. Nothing is ever overwritten:
$\mathbf{x}_k$ holds the input and the outputs of all $k$ layers so far, so
every layer is connected to all its predecessors, hence the name.
the figure shows a dense block with two layers.

![A dense block with two layers and growth rate 32. Each layer appends its 32 output channels to its input, so a 64-channel input grows to 96 and then 128 channels, and every layer sees the outputs of all layers before it.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-densenet-block.svg)

Each convolution in a dense block is preceded by batch normalization and a
ReLU, the "normalization, activation, convolution" order that the exercises
explore for ResNet. The convolutions have no bias, since every layer that
reads their output starts with batch normalization. The number of channels
that each layer appends, 32 in the figure, is called the
*growth rate*. Since every dense block adds channels, a stack of them would
soon be very wide. Between dense blocks, a *transition layer* therefore
reduces the number of channels with a $1 \times 1$ convolution and halves
the height and the width with average pooling of stride 2. A full DenseNet
has four dense blocks separated by transition layers, like the four stages
of ResNet-18; problem 9 builds and trains one.

DenseNet reached the ImageNet accuracy of ResNet with fewer parameters,
which its authors attribute to the reuse of features. Unfortunately,
concatenation costs memory. The distinct feature
maps of a dense block grow linearly with its depth, but a direct
implementation materializes a new, wider concatenation for every layer and
keeps it, with its normalized version, for the backward pass, so its memory
grows quadratically with the depth of the block. Memory-efficient
implementations recompute these tensors during backpropagation
[@pleiss2017memory]. Addition has no such cost, which is one reason why
residual connections, not dense ones, became the standard in later
architectures.

Concatenating skip connections did survive in one widely used design. The
U-Net [@Ronneberger.Fischer.Brox.2015], a network for image
segmentation, first reduces the resolution step by step and then restores
it. On the way up it concatenates each upsampled feature map with the
feature map of the same resolution from the way down. It does so once per
resolution rather than after every layer, so its memory stays modest. The
denoising networks of diffusion models are built on it
(that section).

## Summary

Deeper plain networks can fit even their training data worse than shallower
ones, although they could represent the same functions; why training fails
to find them is not fully understood. A residual block computes its input
plus a learned correction, which moves the identity from particular weight
values to the point of zero correction and gives the gradient a direct path
to every block. On Fashion-MNIST, a plain network with 56 layers ends
training with a higher training loss than one with 20 layers, while with
residual blocks the deeper network ends with the lower one. Stacked in four
stages with batch normalization, residual blocks give ResNet-18, which
reaches about 93% validation accuracy on Fashion-MNIST in 10 epochs, and the
same template scales to 152 layers.
ResNeXt replaces the middle convolution of a bottleneck block by a grouped
convolution, which makes the block wider at the same cost, and DenseNet
concatenates the outputs of its layers instead of adding them, which reuses
features but needs more memory. Residual connections went on to become a
part of nearly every deep architecture, including the transformers of
that section.

## Exercises

1. **Inception versus residual blocks.** In the `arch` of
   GoogLeNet (that section), the second block of the middle group has
   branch widths $c_1 = 160$, $c_2 = (112, 224)$, $c_3 = (24, 64)$, and
   $c_4 = 64$, and it maps 512 input channels to 512 output channels.
   Compare it with a 512-channel `Residual` block that uses an identity
   shortcut.
    1. Count the convolution weights and the multiply-adds per output
       pixel of both blocks, ignoring biases and normalization
       parameters.
    1. A `Residual` block whose residual branch outputs zero is the
       identity on nonnegative inputs. Determine whether any choice of
       parameters makes the Inception block the identity on nonnegative
       inputs, and state what the answer implies for deepening GoogLeNet
       by inserting Inception blocks.

1. **Pre-activation ordering.** @He.Zhang.Ren.ea.2016*1
   moved batch normalization and the ReLU in front of each convolution of
   the residual branch and removed the ReLU after the addition (their
   Figure 1), so that a block computes $\mathbf{x} + g(\mathbf{x})$ with
   the identity path left untouched. The DenseNet convolution block in
   that section uses this "normalization, activation,
   convolution" order.
    1. Stack $K$ pre-activation blocks,
       $\mathbf{x}_k = \mathbf{x}_{k-1} + g_k(\mathbf{x}_{k-1})$ for
       $k = 1, \ldots, K$. Write $\mathbf{x}_K$ in terms of
       $\mathbf{x}_0$ and the branch outputs, and show that the Jacobian
       $\partial \mathbf{x}_K / \partial \mathbf{x}_0$ contains the
       identity matrix as an additive term.
    1. Now stack $K$ `Residual` blocks with identity shortcuts, each of
       which applies a ReLU after the addition. Which factor takes the
       place of the identity term in
       $\partial \mathbf{x}_K / \partial \mathbf{x}_0$, and what does it
       imply for the gradient that reaches the first block through the
       shortcuts?

1. **Nested function classes.** The containment argument of
   the figure applies to a deeper network when each
   added block can represent the identity on every input it receives.
   Every `Residual` block in `ResNet18` receives nonnegative inputs.
    1. Set the weights of `conv2` in a `Residual` block with an identity
       shortcut to zero. Determine the output of `bn2` in
       training mode and in prediction mode, and decide in each mode
       whether the block is then the identity on nonnegative inputs.
       Name parameter values of `bn2` that make the block the identity
       in both modes.
    1. ResNet-34 (Table 1 of @He.Zhang.Ren.ea.2016) has the same
       stage widths as `ResNet18` and more blocks per stage. Decide
       whether ResNet-34 represents every function that `ResNet18`
       represents, and justify the answer block by block.

1. [code] **Gradients of plain networks.** ● According to
   @He.Zhang.Ren.ea.2016, the gradients of their plain networks
   kept a healthy size. Check this on the four networks that
   that section trains.
    1. At initialization, compute the gradient of the loss on one minibatch
       and plot the norm of the gradient of each convolutional layer's
       weights against the index of the layer, for all four networks. Do
       the gradients of the 56-layer plain network shrink toward the input?
    1. Repeat the measurement after one epoch of training.
    1. Train the 56-layer plain network for 20 epochs instead of 5. Does
       its training loss catch up with that of the 20-layer plain network
       trained equally long? What does the answer say about whether the
       deep plain network is slow or stuck?

1. **Paths through a residual stack.** A stack of $K$ residual blocks can
   be unraveled into an ensemble of paths: on each block, the signal
   either passes through the block's body or skips it
   [@Veit.Wilber.Belongie.2016].
    1. How many end-to-end paths pass through exactly $\ell$ of the $K$
       bodies? Verify your formula by explicit enumeration for $K = 4$.
    1. Relate the distribution of path lengths to why very deep residual
       networks remain trainable where equally deep plain networks are
       not.

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problem 11.2.*

1. [code] **ResNet variants.** ● Table 1 of @He.Zhang.Ren.ea.2016
   specifies ResNet-18 to ResNet-152 by the number of blocks per stage and
   by the block type.
    1. Build ResNet-34 by passing its block counts as `arch` to `ResNet`
       and report its parameter count against `ResNet18`.
    1. ResNet-50 and its deeper relatives use a bottleneck block: a
       $1 \times 1$ convolution that reduces the width fourfold, a
       $3 \times 3$ convolution, and a $1 \times 1$ convolution that
       restores the width. `ResNeXtBlock` with `groups=1` and
       `bot_mul=0.25` implements this block. Compare its parameter count
       and multiply-adds with those of `Residual` at the same input and
       output width, and with those of `Residual` at a quarter of that
       width, which is the width of the matching ResNet-34 stage.
    1. Override `stage` in a subclass of `ResNet` so that it stacks
       bottleneck blocks, build ResNet-50, and compare its parameter count
       with ResNet-34's.

1. [code] **ResNet-D shortcut.** In a downsampling `Residual` block, the
   stride-2 $1 \times 1$ convolution on the shortcut reads one of every
   four input positions. @He.Zhang.Zhang.ea.2019 replace it by
   $2 \times 2$ average pooling with stride 2, followed by a stride-1
   $1 \times 1$ convolution and batch normalization.
    1. Implement this shortcut in a subclass of `Residual` and check on a
       random input that the new block has the same output shape and the
       same number of parameters as the original one.
    1. Build a ResNet-18 from the new block and train it with the settings
       of this section. Compare its validation accuracy with that of
       `ResNet18` over several runs of each network. Is the difference
       larger than the variation between runs?

1. [code] **Pre-activation ResNet-18.** ● Implement the pre-activation
   block of @He.Zhang.Ren.ea.2016*1 as a variant of `Residual`,
   override `stage` in a subclass of `ResNet` so that it stacks the new
   block, and compare the training curves of the resulting ResNet-18 with
   those of `ResNet18`.

1. [code] **DenseNet.** ● Build and train the DenseNet of
   that section, then account for its parameters and its memory.
    1. Implement the convolution block of a dense block (batch
       normalization, a ReLU, and a $3 \times 3$ convolution without bias),
       a `DenseBlock` that applies `num_convs` such blocks and concatenates
       the output of each one to its input along the channel axis, and a
       transition layer (batch normalization, a ReLU, a $1 \times 1$
       convolution without bias, and $2 \times 2$ average pooling with
       stride 2). Check on a random input with 3 channels that a dense
       block with two convolutions and growth rate 10 returns
       $3 + 10 + 10 = 23$ channels, and that a transition layer with 10
       channels reduces them to 10 and halves the height and the width.
    1. Assemble a DenseNet from the stem of `ResNet`, four dense blocks of
       four convolutions each with growth rate 32, a transition layer that
       halves the number of channels between consecutive dense blocks, and
       batch normalization and a ReLU before the head of `ResNet`. Train it
       with the settings of this section and compare its parameter count,
       validation accuracy, and time per epoch with those of `ResNet18`.
    1. @Huang.Liu.Van-Der-Maaten.ea.2017 report that DenseNets
       match the accuracy of comparable ResNets with fewer parameters,
       whereas @pleiss2017memory had to devise memory-efficient
       implementations for them. Consider a dense block whose $n$
       convolutions each emit $r$ channels on an input with $c$ channels,
       and compare it with $n$ `Residual` blocks of constant width $c$.
       Ignore biases and batch normalization parameters. Your `DenseBlock`
       builds the input of each convolution as a new concatenated tensor,
       and a naive implementation keeps all of these inputs for the
       backward pass. How many channels does the $i$-th convolution
       consume? Sum them over the block and compare the sum with the $2n$
       inputs of width $c$ that the $3 \times 3$ convolutions of the
       residual blocks keep.
    1. Count the weights of the dense block's $3 \times 3$ convolutions
       and compare them with those of the $n$ residual blocks. For which
       growth rates $r$ does the dense block use fewer parameters?
    1. @pleiss2017memory store only the $c + nr$ channels of
       distinct feature maps and discard each concatenated tensor and its
       normalized version after use. For $c = 64$, $r = 32$, and $n = 6$,
       compare this storage with your sum from the third part. Explain
       why the backward pass must then recompute concatenations, batch
       normalizations, and ReLUs, but no convolution.

[Discussions](https://d2l.discourse.group/t/18006)